In [175]:
!pip install datasets   # Load dataset

In [176]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [177]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [178]:
import pandas as pd

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(train_df.head())

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0


In [179]:
print("Train size:", len(train_df))
print("Test size:", len(test_df))

print("\nTrain label dağılımı:")
print(train_df["label"].value_counts())

Train size: 25000
Test size: 25000

Train label dağılımı:
label
0    12500
1    12500
Name: count, dtype: int64


In [180]:
train_df = train_df.sample(5000, random_state=42)
test_df = test_df.sample(1000, random_state=42)

print("Yeni train size:", len(train_df))
print("Yeni test size:", len(test_df))

Yeni train size: 5000
Yeni test size: 1000


In [181]:
import re

def clean_text(text):
    text = text.lower()                        # küçük harfe çevir
    text = re.sub(r'<.*?>', '', text)          # <br /> gibi HTML tagları sil
    text = re.sub(r'[^a-z0-9\s]', '', text)   # noktalama işaretlerini sil
    text = re.sub(r'\s+', ' ', text).strip()  # fazla boşlukları sil
    return text

train_df['text'] = train_df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)

print("Temizleme tamamlandı!")
print("\nÖrnek temizlenmiş metin:")
print(train_df['text'].iloc[0][:300])

Temizleme tamamlandı!

Örnek temizlenmiş metin:
dumb is as dumb does in this thoroughly uninteresting supposed black comedy essentially what starts out as chris klein trying to maintain a low profile eventually morphs into an uninspired version of the three amigos only without any laughs in order for black comedy to work it must be outrageous whi


In [182]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

y_train = train_df["label"]
y_test = test_df["label"]

print("Geliştirilmiş TF-IDF tamamlandı")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Geliştirilmiş TF-IDF tamamlandı
X_train shape: (5000, 10000)
X_test shape: (1000, 10000)


In [183]:
import torch

# sparse matrix → normal array → tensor
X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print("Tensor dönüşümü tamam")
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

Tensor dönüşümü tamam
X_train_tensor: torch.Size([5000, 10000])
y_train_tensor: torch.Size([5000, 1])


In [184]:
import torch.nn as nn

class SentimentNet(nn.Module):
    def __init__(self, input_size):
        super(SentimentNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

input_size = X_train_tensor.shape[1]
model = SentimentNet(input_size)

print(model)

SentimentNet(
  (fc1): Linear(in_features=10000, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [185]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Hazır")

Hazır


In [186]:
from torch.utils.data import DataLoader, TensorDataset

# Dataset ve DataLoader oluştur
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/20, Loss: 0.4570
Epoch 2/20, Loss: 0.1043
Epoch 3/20, Loss: 0.0214
Epoch 4/20, Loss: 0.0035
Epoch 5/20, Loss: 0.0007
Epoch 6/20, Loss: 0.0003
Epoch 7/20, Loss: 0.0001
Epoch 8/20, Loss: 0.0000
Epoch 9/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0000
Epoch 11/20, Loss: 0.0000
Epoch 12/20, Loss: 0.0000
Epoch 13/20, Loss: 0.0000
Epoch 14/20, Loss: 0.0000
Epoch 15/20, Loss: 0.0000
Epoch 16/20, Loss: 0.0000
Epoch 17/20, Loss: 0.0000
Epoch 18/20, Loss: 0.0000
Epoch 19/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000


In [187]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    predictions = (test_outputs >= 0.5).float()

accuracy = accuracy_score(y_test_tensor, predictions)
print("Test Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test_tensor, predictions))

Test Accuracy: 0.834

Classification Report:
              precision    recall  f1-score   support

         0.0       0.83      0.85      0.84       511
         1.0       0.84      0.82      0.83       489

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000



In [188]:
!pip install gradio

In [189]:
import gradio as gr

def predict_sentiment(text):
    # Temizle
    cleaned = clean_text(text)

    # TF-IDF'e çevir
    vectorized = vectorizer.transform([cleaned])
    tensor = torch.tensor(vectorized.toarray(), dtype=torch.float32)

    # Tahmin yap
    model.eval()
    with torch.no_grad():
        output = model(tensor)
        probability = output.item()
        prediction = "Positive 😊" if probability >= 0.5 else "Negative 😞"
        confidence = probability if probability >= 0.5 else 1 - probability

    return f"{prediction} (Confidence: {confidence:.1%})"

# Gradio arayüzü
app = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Write a movie review here...",
        label="Movie Review"
    ),
    outputs=gr.Textbox(label="Sentiment"),
    title="🎬 Sentiment Analysis App",
    description="Enter a movie review and the model will predict if it's Positive or Negative."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a851828f65ec2187e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
